In [0]:
from pyspark.sql import functions as F

# ---------------- CONFIG ----------------

BRONZE_JIRA = "workspace.slainte_bronze.jira_issues_bronze"

GOLD_DB     = "workspace.slainte_gold"

DIM_PRIORITY = f"{GOLD_DB}.dim_priority"

spark.sql(f"CREATE DATABASE IF NOT EXISTS {GOLD_DB}")

# ---------------- BUILD DIM ----------------

df_dim_priority = (

    spark.table(BRONZE_JIRA)

    .select(

        F.col("user_id").alias("source_user_id"),

        F.col("project_key").alias("project"),

        F.col("priority").alias("priority_code"),

        F.col("priority_label")   # ✅ already computed in bronze

    )

    .filter(

        F.col("source_user_id").isNotNull() &

        F.col("priority_code").isNotNull()

    )

    .distinct()

)

# ---------------- SAFETY (ensure label exists) ----------------

df_dim_priority = df_dim_priority.withColumn(

    "priority_label",

    F.coalesce(F.col("priority_label"), F.lit(99))

)

# ---------------- ADD ID ----------------

df_dim_priority = (

    df_dim_priority

    .withColumn("priority_id", F.monotonically_increasing_id() + 1)

    .select(

        "priority_id",

        "source_user_id",

        "project",

        "priority_code",

        "priority_label"   # ✅ IMPORTANT

    )

)

# ---------------- WRITE ----------------

df_dim_priority.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(DIM_PRIORITY)

print("✅ dim_priority updated with priority_label")
 

In [0]:
from pyspark.sql import functions as F

from delta.tables import DeltaTable

# ---------------- CONFIG ----------------

BRONZE_JIRA = "workspace.slainte_bronze.jira_issues_bronze"

GOLD_DB     = "workspace.slainte_gold"

DIM_PRIORITY = f"{GOLD_DB}.dim_priority"

spark.sql(f"CREATE DATABASE IF NOT EXISTS {GOLD_DB}")

# ---------------- BUILD DIM ----------------

df_dim_priority = (

    spark.table(BRONZE_JIRA)

    .select(

        F.col("user_id").alias("source_user_id"),

        F.col("project_key").alias("project"),

        F.col("priority").alias("priority_code"),

        F.col("priority_label")

    )

    .filter(

        F.col("source_user_id").isNotNull() &

        F.col("priority_code").isNotNull()

    )

    .distinct()

)

# ---------------- SAFETY ----------------

df_dim_priority = df_dim_priority.withColumn(

    "priority_label",

    F.coalesce(F.col("priority_label"), F.lit(99))

)

# ---------------- ADD ID ----------------

df_dim_priority = (

    df_dim_priority

    .withColumn("priority_id", F.monotonically_increasing_id() + 1)

    .select(

        "priority_id",

        "source_user_id",

        "project",

        "priority_code",

        "priority_label"

    )

)

# ---------------- WRITE (MERGE) ----------------

if not spark.catalog.tableExists(DIM_PRIORITY):

    df_dim_priority.write.format("delta") \
        .mode("overwrite") \
        .option("overwriteSchema", "true") \
        .saveAsTable(DIM_PRIORITY)

else:

    target = DeltaTable.forName(spark, DIM_PRIORITY)

    (

        target.alias("t")

        .merge(

            df_dim_priority.alias("s"),

            "t.source_user_id = s.source_user_id AND "

            "t.project = s.project AND "

            "t.priority_code = s.priority_code"

        )

        .whenMatchedUpdateAll()

        .whenNotMatchedInsertAll()

        .execute()

    )

print("✅ dim_priority merged successfully")
 